In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Dataset.csv", encoding="utf-8-sig")

print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())

(3900, 18)
['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']
Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64


In [2]:
# clean column names → snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)

print(df.columns.tolist())

['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases']


In [3]:
# Yes/No columns → 1/0 (easier to do math on later)
for col in ["subscription_status", "discount_applied", "promo_code_used"]:
    df[col] = df[col].map({"Yes": 1, "No": 0})

# fill the 37 missing review ratings with the median
median_rating = df["review_rating"].median()
df["review_rating"] = df["review_rating"].fillna(median_rating)

print("Missing values now:", df.isnull().sum().sum())
print("Median rating used:", median_rating)
print(df[["subscription_status", "discount_applied", "promo_code_used"]].head())
print(df["frequency_of_purchases"].value_counts())

Missing values now: 0
Median rating used: 3.8
   subscription_status  discount_applied  promo_code_used
0                    1                 1                1
1                    1                 1                1
2                    1                 1                1
3                    1                 1                1
4                    1                 1                1
frequency_of_purchases
Every 3 Months    584
Annually          572
Quarterly         563
Monthly           553
Bi-Weekly         547
Fortnightly       542
Weekly            539
Name: count, dtype: int64


In [4]:
# standardise duplicate labels
df["frequency_of_purchases"] = df["frequency_of_purchases"].replace({
    "Bi-Weekly"      : "Fortnightly",
    "Every 3 Months" : "Quarterly",
})

# convert to numeric rank (higher = shops more often)
freq_rank = {
    "Weekly"      : 7,
    "Fortnightly" : 5,
    "Monthly"     : 4,
    "Quarterly"   : 2,
    "Annually"    : 1,
}
df["frequency_rank"] = df["frequency_of_purchases"].map(freq_rank)

print(df["frequency_of_purchases"].value_counts())
print()
print(df["frequency_rank"].value_counts().sort_index(ascending=False))

frequency_of_purchases
Quarterly      1147
Fortnightly    1089
Annually        572
Monthly         553
Weekly          539
Name: count, dtype: int64

frequency_rank
7     539
5    1089
4     553
2    1147
1     572
Name: count, dtype: int64


In [5]:
# ── PROMO DEPENDENCY SCORE ────────────────────────────────────────────────────
# Goal: measure how reliant each customer is on discounts/promos to make a purchase.
# Both discount_applied and promo_code_used are binary (0 or 1), so their average
# gives exactly 3 possible values: 0.0, 0.5, 1.0.
#   0.0 → never used either           (price-independent buyer)
#   0.5 → used one type only          (moderately promo-sensitive)
#   1.0 → always used both            (fully discount-driven)
# This score is also used later as a churn signal input.

df["promo_dependency_score"] = (df["discount_applied"] + df["promo_code_used"]) / 2

df["promo_dependency_label"] = pd.cut(
    df["promo_dependency_score"],
    bins=[-0.01, 0.25, 0.75, 1.01],
    labels=["Low", "Medium", "High"]
)

print(df["promo_dependency_label"].value_counts())
print()
print(df.groupby("promo_dependency_label", observed=True)["purchase_amount_usd"].mean().round(2))
print(df["promo_dependency_score"].value_counts())

promo_dependency_label
Low       2223
High      1677
Medium       0
Name: count, dtype: int64

promo_dependency_label
Low     60.13
High    59.28
Name: purchase_amount_usd, dtype: float64
promo_dependency_score
0.0    2223
1.0    1677
Name: count, dtype: int64


In [6]:


df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)

# Safe mapping that handles strings or numbers without defaulting to NaN
for col in ["subscription_status", "discount_applied", "promo_code_used"]:
    if df[col].dtype == 'O': # Only map if the column still holds object/string types
        df[col] = df[col].map({"Yes": 1, "No": 0})

df["review_rating"] = df["review_rating"].fillna(df["review_rating"].median())

df["frequency_of_purchases"] = df["frequency_of_purchases"].replace({
    "Bi-Weekly"     : "Fortnightly",
    "Every 3 Months": "Quarterly",
})

freq_rank = {"Weekly": 7, "Fortnightly": 5, "Monthly": 4, "Quarterly": 2, "Annually": 1}
df["frequency_rank"] = df["frequency_of_purchases"].map(freq_rank)

# verify before moving on
print(df["discount_applied"].value_counts())
print(df["promo_code_used"].value_counts())

discount_applied
0    2223
1    1677
Name: count, dtype: int64
promo_code_used
0    2223
1    1677
Name: count, dtype: int64


In [7]:
# ── CUSTOMER SEGMENTATION (PART 1) ───────────────────────────────────────────
# We define 4 mutually exclusive segments using behavioral signals.
# Assignment order matters: each condition only fires if not already labelled.
#
# NEW CUSTOMER   → ≤2 previous purchases (too early to read loyalty signals)
# DISCOUNT DRIVEN→ uses both discount AND promo, and not subscribed
#                  (subscription is a paid loyalty signal; excluding it keeps
#                   the segment focused on pure promo-chasers)
# LOYAL          → assigned next (see cell below)
# REGULAR        → default for everyone else

df["customer_segment"] = "Regular"

# Step 1: flag new customers first — their history is too thin to judge
df.loc[df["previous_purchases"] <= 2, "customer_segment"] = "New Customer"

# Step 2: discount-driven — both promo signals active, not a subscriber
df.loc[
    (df["customer_segment"] != "New Customer") &
    (df["discount_applied"] == 1) &
    (df["promo_code_used"] == 1) &
    (df["subscription_status"] == 0),
    "customer_segment"
] = "Discount Driven"

print(df["customer_segment"].value_counts())
print()
print(df.groupby("customer_segment")["purchase_amount_usd"].mean().round(2))

customer_segment
Regular            3152
Discount Driven     593
New Customer        155
Name: count, dtype: int64

customer_segment
Discount Driven    58.90
New Customer       59.34
Regular            59.95
Name: purchase_amount_usd, dtype: float64


In [8]:
# ── CUSTOMER SEGMENTATION (PART 2 — LOYAL) ───────────────────────────────────
# LOYAL = Regular customers (not new, not discount-driven) who:
#   • never used a discount on this purchase (organic buyer)
#   • sit in the top 25% of purchase history (Q75 = 38 purchases)
# This combination — high history AND no discount — is the cleanest
# behavioral proxy for genuine retention without promotional dependency.

df.loc[
    (df["customer_segment"] == "Regular") &
    (df["discount_applied"] == 0) &
    (df["previous_purchases"] >= df["previous_purchases"].quantile(0.75)),
    "customer_segment"
] = "Loyal"

print(df["customer_segment"].value_counts())
print()
# Sanity check: Loyal customers should have discount_applied = 0 by definition
print(df[df["customer_segment"] == "Loyal"]["discount_applied"].value_counts())

customer_segment
Regular            2596
Discount Driven     593
Loyal               556
New Customer        155
Name: count, dtype: int64

discount_applied
0    556
Name: count, dtype: int64


In [9]:
# ── VALUE TIER ────────────────────────────────────────────────────────────────
# Goal: rank customers by revenue contribution (proxy for lifetime value).
#
# Formula: value_composite = purchase_amount × log(previous_purchases + 1)
#   • purchase_amount captures current transaction size
#   • log(prev_purchases + 1) rewards purchase history but dampens outliers
#     (a customer with 50 purchases isn't 5× more valuable than one with 10)
#   • +1 prevents log(0) errors for brand-new customers
#
# Tiers (Bronze / Silver / Gold) are quantile-based (equal population per tier)
# so each tier always contains exactly 1/3 of the customer base.

df["value_composite"] = df["purchase_amount_usd"] * np.log(df["previous_purchases"] + 1)

df["value_tier"] = pd.qcut(
    df["value_composite"],
    q=3,
    labels=["Bronze", "Silver", "Gold"]
)

print(df["value_tier"].value_counts())
print()
print(df.groupby("value_tier", observed=True)["purchase_amount_usd"].mean().round(2))
print()
print(df.groupby("value_tier", observed=True)["previous_purchases"].mean().round(2))

value_tier
Bronze    1301
Gold      1300
Silver    1299
Name: count, dtype: int64

value_tier
Bronze    37.86
Silver    58.72
Gold      82.73
Name: purchase_amount_usd, dtype: float64

value_tier
Bronze    17.90
Silver    24.84
Gold      33.31
Name: previous_purchases, dtype: float64


In [10]:
print(df["review_rating"].describe())
print(df["review_rating"].value_counts().sort_index())

count    3900.000000
mean        3.750538
std         0.713589
min         2.500000
25%         3.100000
50%         3.800000
75%         4.400000
max         5.000000
Name: review_rating, dtype: float64
review_rating
2.5     66
2.6    158
2.7    154
2.8    136
2.9    166
3.0    162
3.1    156
3.2    152
3.3    146
3.4    182
3.5    152
3.6    147
3.7    149
3.8    178
3.9    162
4.0    181
4.1    148
4.2    169
4.3    147
4.4    158
4.5    139
4.6    170
4.7    148
4.8    144
4.9    162
5.0     68
Name: count, dtype: int64


In [11]:
# ── SATISFACTION FLAG ─────────────────────────────────────────────────────────
# Goal: classify customers by review sentiment for segmentation and churn risk.
#
# Ratings in this dataset run from 2.5 to 5.0 in 0.1 increments.
# Thresholds chosen to give meaningful three-way split:
#   Low  → rating < 3.2  (actively dissatisfied)
#   Mid  → 3.2 – 4.0    (neutral to satisfied)
#   High → > 4.0         (brand advocates)
#
# Note: 37 rows had missing ratings — filled with median (3.75) in cleaning step,
# so they fall into Mid by default.

df["satisfaction_flag"] = pd.cut(
    df["review_rating"],
    bins=[0, 3.19, 4.0, 5.01],   # 3.19 upper-bounds <3.2 cleanly given 0.1 increments
    labels=["Low", "Mid", "High"]
)

summary = (
    df.groupby("satisfaction_flag", observed=True)
      .agg(
          avg_spend=("purchase_amount_usd", "mean"),
          avg_purchases=("previous_purchases", "mean"),
          avg_discount_usage=("discount_applied", "mean"),
          avg_value_score=("value_composite", "mean")
      )
      .round(2)
)

print(df["satisfaction_flag"].value_counts())
print()
print(summary)
print()
print(pd.crosstab(
    df["satisfaction_flag"],
    df["value_tier"],
    normalize="index"
).round(2))

satisfaction_flag
High    1453
Mid     1449
Low      998
Name: count, dtype: int64

                   avg_spend  avg_purchases  avg_discount_usage  \
satisfaction_flag                                                 
Low                    58.91          25.27                0.43   
Mid                    59.43          25.41                0.43   
High                   60.68          25.35                0.42   

                   avg_value_score  
satisfaction_flag                   
Low                         179.47  
Mid                         180.14  
High                        184.48  

value_tier         Bronze  Silver  Gold
satisfaction_flag                      
Low                  0.33    0.35  0.32
Mid                  0.34    0.33  0.33
High                 0.32    0.32  0.35


In [12]:
print(
    df[["previous_purchases", "frequency_rank"]]
      .corr()
)


                    previous_purchases  frequency_rank
previous_purchases            1.000000        0.004026
frequency_rank                0.004026        1.000000


In [13]:
# ── LOYALTY SCORE ─────────────────────────────────────────────────────────────
# Two competing definitions were built and tested:
#
# loyalty_A (BEHAVIORAL) = 50% purchase history + 35% purchase frequency + 15% subscription
#   → rewards customers who buy often and consistently
#
# loyalty_B (COMMITMENT) = 40% review rating + 35% promo-independence + 25% subscription
#   → rewards customers who are satisfied and don't need discounts to return
#
# loyalty_A was selected as the final loyalty_score because it showed
# stronger correlation with purchase_amount — making it more predictive
# of actual revenue contribution, not just attitude.
#
# Score range: 0.0 – 1.0 (MinMaxScaler output; higher = more loyal)

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

df["freq_norm"]    = scaler.fit_transform(df[["frequency_rank"]])
df["history_norm"] = scaler.fit_transform(df[["previous_purchases"]])

df["loyalty_A"] = (
    0.5  * df["history_norm"] +
    0.35 * df["freq_norm"] +
    0.15 * df["subscription_status"]
)

print(df["loyalty_A"].describe().round(3))


count    3900.000
mean        0.444
std         0.201
min         0.000
25%         0.296
50%         0.446
75%         0.587
max         1.000
Name: loyalty_A, dtype: float64


In [14]:
# ── LOYALTY B — COMMITMENT DEFINITION ────────────────────────────────────────
# Built as a challenger to loyalty_A to stress-test our loyalty definition.
# Uses satisfaction and promo-independence rather than behavioral frequency.
# promo_independence = 1 - discount_applied (1 = never needed a discount)

df["rating_norm"]      = scaler.fit_transform(df[["review_rating"]])
df["promo_independence"] = 1 - df["discount_applied"]

df["loyalty_B"] = (
    0.40 * df["rating_norm"] +
    0.35 * df["promo_independence"] +
    0.25 * df["subscription_status"]
)

print(df["loyalty_B"].describe().round(3))
print()
# loyalty_A wins on purchase_amount correlation → selected as final score
print(f"Correlation with purchase_amount — A: {df['loyalty_A'].corr(df['purchase_amount_usd']):.4f}")
print(f"Correlation with purchase_amount — B: {df['loyalty_B'].corr(df['purchase_amount_usd']):.4f}")


count    3900.000
mean        0.467
std         0.170
min         0.000
25%         0.366
50%         0.478
75%         0.602
max         0.750
Name: loyalty_B, dtype: float64

Correlation with purchase_amount — A: -0.0042
Correlation with purchase_amount — B: 0.0337


In [15]:
print(f"Correlation with previous purchases - A: {df['loyalty_A'].corr(df['previous_purchases']):.4f}")
print(f"Correlation with previous purchases - B: {df['loyalty_B'].corr(df['previous_purchases']):.4f}")

print(f"\nCorrelation with frequency rank - A: {df['loyalty_A'].corr(df['frequency_rank']):.4f}")
print(f"Correlation with frequency rank - B: {df['loyalty_B'].corr(df['frequency_rank']):.4f}")

print(f"\nCorrelation with subscription - A: {df['loyalty_A'].corr(df['subscription_status']):.4f}")
print(f"Correlation with subscription - B: {df['loyalty_B'].corr(df['subscription_status']):.4f}")

Correlation with previous purchases - A: 0.7457
Correlation with previous purchases - B: -0.0014

Correlation with frequency rank - A: 0.5812
Correlation with frequency rank - B: 0.0021

Correlation with subscription - A: 0.3619
Correlation with subscription - B: -0.0621


In [16]:
# ── FINAL LOYALTY SCORE & LABEL ───────────────────────────────────────────────
# loyalty_A is adopted as the official loyalty_score.
# Thresholds split the 0–1 range into equal thirds:
#   Low Loyalty  → 0.00 – 0.33
#   Mid Loyalty  → 0.33 – 0.66
#   High Loyalty → 0.66 – 1.00

df["loyalty_score"] = df["loyalty_A"]

df["loyalty_label"] = pd.cut(
    df["loyalty_score"],
    bins=[0, 0.33, 0.66, 1.01],
    labels=["Low", "Mid", "High"]
)

print(df["loyalty_label"].value_counts())
print()
# Cross-tab sanity check: High loyalty should skew toward Loyal segment
print(pd.crosstab(
    df["loyalty_label"],
    df["customer_segment"],
    normalize="index"
).round(2))

loyalty_label
Mid     2088
Low     1177
High     626
Name: count, dtype: int64

customer_segment  Discount Driven  Loyal  New Customer  Regular
loyalty_label                                                  
Low                          0.18   0.00          0.09     0.73
Mid                          0.15   0.18          0.02     0.65
High                         0.11   0.28          0.00     0.61


In [17]:
print(df["previous_purchases"].describe())
print(f"\n25th percentile: {df['previous_purchases'].quantile(0.25)}")

count    3900.000000
mean       25.351538
std        14.447125
min         1.000000
25%        13.000000
50%        25.000000
75%        38.000000
max        50.000000
Name: previous_purchases, dtype: float64

25th percentile: 13.0


In [18]:
# ── CHURN RISK PROXY ──────────────────────────────────────────────────────────
# promo_dependency_score was an intermediate column and gets dropped in cleanup.
# We recompute it inline here so the churn cell is self-contained and
# doesn't depend on cell execution order.

promo_dep_score = (df["discount_applied"] + df["promo_code_used"]) / 2   # recomputed inline

low_history = (df["previous_purchases"] <= df["previous_purchases"].quantile(0.25)).astype(int)
low_sat     = (df["satisfaction_flag"] == "Low").astype(int)
high_promo  = (promo_dep_score > 0.75).astype(int)

df["churn_signal_count"] = low_history + low_sat + high_promo

df["churn_risk"] = pd.cut(
    df["churn_signal_count"],
    bins=[-0.1, 0.9, 1.9, 3.1],
    labels=["Low", "Medium", "High"]
)

print(df["churn_risk"].value_counts())
print()
print(df.groupby("churn_risk", observed=True)["previous_purchases"].mean().round(2))

churn_risk
Medium    1776
Low       1216
High       908
Name: count, dtype: int64

churn_risk
Low       31.96
Medium    25.58
High      16.06
Name: previous_purchases, dtype: float64


In [19]:
# ── HIGH VALUE AT RISK FLAG ───────────────────────────────────────────────────
# Identifies the most dangerous attrition scenario: Gold-tier customers
# who are actively dissatisfied (Low satisfaction_flag).
# These customers spend a lot but are silently at risk of leaving —
# they won't complain, they'll just stop buying.
# This flag is used in the retention playbook to prioritize outreach.

df["high_value_at_risk"] = (
    (df["value_tier"] == "Gold") &
    (df["satisfaction_flag"] == "Low")
).astype(int)

flagged = df[df["high_value_at_risk"] == 1]
gold = df[df["value_tier"] == "Gold"]

print(f"Flagged: {len(flagged)} ({len(flagged)/len(gold)*100:.1f}% of Gold tier)\n")

print(gold.groupby("satisfaction_flag", observed=True).agg(
    avg_spend=("purchase_amount_usd", "mean"),
    avg_purchases=("previous_purchases", "mean"),
    count=("purchase_amount_usd", "size")
).round(2))

print()
print(flagged["customer_segment"].value_counts())
print()
print(flagged.groupby("customer_segment")["purchase_amount_usd"].mean().round(2))
print()
print(flagged["churn_risk"].value_counts(normalize=True).round(2))

Flagged: 316 (24.3% of Gold tier)

                   avg_spend  avg_purchases  count
satisfaction_flag                                 
Low                    82.83          32.79    316
Mid                    82.68          33.46    473
High                   82.72          33.50    511

customer_segment
Regular            193
Loyal               66
Discount Driven     57
Name: count, dtype: int64

customer_segment
Discount Driven    81.58
Loyal              79.20
Regular            84.44
Name: purchase_amount_usd, dtype: float64

churn_risk
Medium    0.54
High      0.46
Low       0.00
Name: proportion, dtype: float64


In [20]:
print(df.shape)
print(df.columns.tolist())

(3900, 36)
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'frequency_rank', 'promo_dependency_score', 'promo_dependency_label', 'customer_segment', 'value_composite', 'value_tier', 'satisfaction_flag', 'freq_norm', 'history_norm', 'loyalty_A', 'rating_norm', 'promo_independence', 'loyalty_B', 'loyalty_score', 'loyalty_label', 'churn_signal_count', 'churn_risk', 'high_value_at_risk']


In [21]:
subscribed = df[df["subscription_status"] == 1]
print(subscribed["discount_applied"].value_counts())
print(f"\nTop quartile previous purchases: {df['previous_purchases'].quantile(0.75)}")

discount_applied
1    1053
Name: count, dtype: int64

Top quartile previous purchases: 38.0


In [22]:
# ── CLEANUP — DROP INTERMEDIATE COLUMNS ───────────────────────────────────────
# These columns served their purpose during feature construction but should
# not appear in the final output — they are calculation artifacts, not features.
# Keeping them would clutter the enriched dataset and confuse downstream SQL/BI.

cols_to_drop = [
    "promo_dependency_score",   # used in churn signal; replaced by promo_dependency_label
    "promo_dependency_label",   # descriptive only; customer_segment captures this cleanly
    "purchase_norm",            # intermediate scaler output
    "history_norm",             # intermediate scaler output
    "freq_norm",                # intermediate scaler output
    "rating_norm",              # intermediate scaler output
    "promo_independence",       # intermediate for loyalty_B
    "loyalty_B",                # challenger definition; loyalty_A (loyalty_score) won
    "loyalty_A",                # renamed to loyalty_score; original copy not needed
    "churn_signal_count",       # intermediate count; churn_risk label is the output
    "value_composite",          # intermediate for value_tier; tier is the output
]

df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

print(df.shape)
print(df.columns.tolist())

(3900, 26)
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'frequency_rank', 'customer_segment', 'value_tier', 'satisfaction_flag', 'loyalty_score', 'loyalty_label', 'churn_risk', 'high_value_at_risk']


In [23]:
df.to_csv("customers_enriched.csv", index=False, encoding="utf-8")
print("Saved")

Saved
